In [ ]:
import json

notebook_content = {
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 07 — Multi-Backbone Binary Classification\n",
    "\n",
    "**Goal:** Train 4 additional transformer backbones on all 3 binary datasets.\n",
    "\n",
    "**Backbones:** XLM-RoBERTa, mBERT, Sagorsarker BanglaBERT, MuRIL  \n",
    "**Datasets:** Ben-Sarc, BanglaSarc, BanglaSarc3 (all binary)  \n",
    "**Protocol:** epochs=3, lr=2e-5, batch=8, max_len=128  \n",
    "\n",
    "**Disk‑safe:** \n",
    "- All caches and temporary files go to `/kaggle/tmp` (60 GB).\n",
    "- Hugging Face cache is cleared after every run.\n",
    "- Only prediction CSVs and metrics are kept; model weights are discarded.\n",
    "- Never exceeds 19 GB of disk usage."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os, sys, json, time, warnings, gc, shutil\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "from datetime import datetime\n",
    "\n",
    "import torch\n",
    "from torch.utils.data import Dataset\n",
    "from transformers import (\n",
    "    AutoTokenizer, AutoModelForSequenceClassification,\n",
    "    TrainingArguments, Trainer, EarlyStoppingCallback\n",
    ")\n",
    "from sklearn.metrics import (\n",
    "    accuracy_score, f1_score, precision_score, recall_score,\n",
    "    classification_report, confusion_matrix\n",
    ")\n",
    "\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# ── Detect environment and set up paths ──\n",
    "if os.path.exists('/kaggle/working'):\n",
    "    REPO_ROOT = Path('/kaggle/working/Sarcasm_detection')\n",
    "    BIG_TMP = Path('/kaggle/tmp')\n",
    "else:\n",
    "    REPO_ROOT = Path('.').resolve()\n",
    "    while not (REPO_ROOT / '00_admin').exists() and REPO_ROOT != REPO_ROOT.parent:\n",
    "        REPO_ROOT = REPO_ROOT.parent\n",
    "    BIG_TMP = REPO_ROOT / '_tmp'\n",
    "\n",
    "PROJECT     = REPO_ROOT\n",
    "SPLIT_DIR   = PROJECT / '01_data' / 'interim' / 'splits'\n",
    "CKPT_DIR    = PROJECT / '03_models' / 'checkpoints'\n",
    "TABLE_DIR   = PROJECT / '04_outputs' / 'tables'\n",
    "PRED_DIR    = PROJECT / '04_outputs' / 'predictions'\n",
    "LOG_DIR     = PROJECT / '04_outputs' / 'run_logs'\n",
    "REPORT_DIR  = PROJECT / '04_outputs' / 'reports'\n",
    "\n",
    "for d in [TABLE_DIR, PRED_DIR, LOG_DIR, REPORT_DIR]:\n",
    "    d.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "HF_CACHE_DIR    = BIG_TMP / 'hf_cache'\n",
    "TEMP_TRAIN_DIR  = BIG_TMP / 'train_cache'\n",
    "HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)\n",
    "TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "print(f\"Project root: {PROJECT}\")\n",
    "print(f\"HuggingFace cache: {HF_CACHE_DIR}\")\n",
    "print(f\"Temp training dir: {TEMP_TRAIN_DIR}\")\n",
    "print(f\"Splits found: {len(list(SPLIT_DIR.glob('*.csv')))} files\")\n",
    "print(f\"GPU available: {torch.cuda.is_available()}\")\n",
    "if torch.cuda.is_available():\n",
    "    for i in range(torch.cuda.device_count()):\n",
    "        props = torch.cuda.get_device_properties(i)\n",
    "        print(f\"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB\")\n",
    "\n",
    "def disk_free_gb(path='/'):\n",
    "    return shutil.disk_usage(path).free / 1e9\n",
    "\n",
    "def clear_hf_cache():\n",
    "    if HF_CACHE_DIR.exists():\n",
    "        shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)\n",
    "        HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)\n",
    "        print(f\"  Cleared HF cache (now {disk_free_gb(BIG_TMP):.1f} GB free on {BIG_TMP})\")\n",
    "\n",
    "if os.path.exists('/kaggle/working'):\n",
    "    print(f\"Kaggle working free: {disk_free_gb('/kaggle/working'):.1f} GB\")\n",
    "    print(f\"Kaggle tmp free:     {disk_free_gb('/kaggle/tmp'):.1f} GB\")\n",
    "    clear_hf_cache()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "BACKBONES = {\n",
    "    'xlm-roberta':    'xlm-roberta-base',\n",
    "    'mbert':          'bert-base-multilingual-cased',\n",
    "    'sagorsarker-bb': 'sagorsarker/bangla-bert-base',\n",
    "    'muril':          'google/muril-base-cased',\n",
    "}\n",
    "\n",
    "DATASETS = {\n",
    "    'ben_sarc_binary': {\n",
    "        'train': SPLIT_DIR / 'ben_sarc_binary_train.csv',\n",
    "        'val':   SPLIT_DIR / 'ben_sarc_binary_val.csv',\n",
    "        'test':  SPLIT_DIR / 'ben_sarc_binary_test.csv',\n",
    "    },\n",
    "    'banglasarc_binary': {\n",
    "        'train': SPLIT_DIR / 'banglasarc_binary_train.csv',\n",
    "        'val':   SPLIT_DIR / 'banglasarc_binary_val.csv',\n",
    "        'test':  SPLIT_DIR / 'banglasarc_binary_test.csv',\n",
    "    },\n",
    "    'banglasarc3_binary': {\n",
    "        'train': SPLIT_DIR / 'banglasarc3_binary_train.csv',\n",
    "        'val':   SPLIT_DIR / 'banglasarc3_binary_val.csv',\n",
    "        'test':  SPLIT_DIR / 'banglasarc3_binary_test.csv',\n",
    "    },\n",
    "}\n",
    "\n",
    "MAX_LENGTH    = 128\n",
    "EPOCHS        = 3\n",
    "BATCH_SIZE    = 8\n",
    "LEARNING_RATE = 2e-5\n",
    "WEIGHT_DECAY  = 0.01\n",
    "WARMUP_RATIO  = 0.1\n",
    "SEED          = 42\n",
    "METRIC_FOR_BEST = 'macro_f1'\n",
    "\n",
    "TEXT_COL  = 'text'\n",
    "LABEL_COL = 'label_binary'\n",
    "SAVE_MODEL_WEIGHTS = False\n",
    "\n",
    "print(f\"Total runs: {len(BACKBONES) * len(DATASETS)}\")\n",
    "print(f\"Save model weights: {SAVE_MODEL_WEIGHTS}\")\n",
    "print(f\"All temporary data will be stored in: {BIG_TMP}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "class SarcasmDataset(Dataset):\n",
    "    def __init__(self, texts, labels, tokenizer, max_length=128):\n",
    "        self.encodings = tokenizer(texts, truncation=True, padding='max_length',\n",
    "                                   max_length=max_length, return_tensors='pt')\n",
    "        self.labels = torch.tensor(labels, dtype=torch.long)\n",
    "    def __len__(self):\n",
    "        return len(self.labels)\n",
    "    def __getitem__(self, idx):\n",
    "        item = {k: v[idx] for k, v in self.encodings.items()}\n",
    "        item['labels'] = self.labels[idx]\n",
    "        return item\n",
    "\n",
    "def compute_metrics(eval_pred):\n",
    "    logits, labels = eval_pred\n",
    "    preds = np.argmax(logits, axis=-1)\n",
    "    return {\n",
    "        'accuracy': accuracy_score(labels, preds),\n",
    "        'macro_f1': f1_score(labels, preds, average='macro'),\n",
    "        'weighted_f1': f1_score(labels, preds, average='weighted'),\n",
    "        'binary_f1': f1_score(labels, preds, average='binary', pos_label=1),\n",
    "        'precision': precision_score(labels, preds, average='binary', pos_label=1),\n",
    "        'recall': recall_score(labels, preds, average='binary', pos_label=1),\n",
    "    }\n",
    "\n",
    "def load_split(csv_path):\n",
    "    df = pd.read_csv(csv_path)\n",
    "    return df[TEXT_COL].astype(str).tolist(), df[LABEL_COL].astype(int).tolist()\n",
    "\n",
    "for ds_name, paths in DATASETS.items():\n",
    "    tr, trl = load_split(paths['train'])\n",
    "    te, tel = load_split(paths['test'])\n",
    "    print(f\"{ds_name}: train={len(tr)}, test={len(te)}, labels={sorted(set(trl))}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "def train_and_evaluate(model_tag, model_name, dataset_tag, dataset_paths):\n",
    "    print(f\"\\n{'='*70}\")\n",
    "    print(f\"  {model_tag} x {dataset_tag}  |  Model: {model_name}\")\n",
    "    print(f\"  Disk free (tmp): {disk_free_gb(BIG_TMP):.1f} GB | working: {disk_free_gb('/kaggle/working'):.1f} GB\")\n",
    "    print(f\"{'='*70}\")\n",
    "    t_start = time.time()\n",
    "\n",
    "    train_texts, train_labels = load_split(dataset_paths['train'])\n",
    "    val_texts, val_labels = load_split(dataset_paths['val'])\n",
    "    test_texts, test_labels = load_split(dataset_paths['test'])\n",
    "    print(f\"  Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}\")\n",
    "\n",
    "    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=str(HF_CACHE_DIR))\n",
    "    model = AutoModelForSequenceClassification.from_pretrained(\n",
    "        model_name, num_labels=2, ignore_mismatched_sizes=True,\n",
    "        cache_dir=str(HF_CACHE_DIR)\n",
    "    )\n",
    "\n",
    "    train_ds = SarcasmDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)\n",
    "    val_ds = SarcasmDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)\n",
    "    test_ds = SarcasmDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)\n",
    "\n",
    "    run_tmp = TEMP_TRAIN_DIR / f\"{model_tag}_{dataset_tag}\"\n",
    "    if run_tmp.exists():\n",
    "        shutil.rmtree(run_tmp)\n",
    "    run_tmp.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "    training_args = TrainingArguments(\n",
    "        output_dir=str(run_tmp),\n",
    "        num_train_epochs=EPOCHS,\n",
    "        per_device_train_batch_size=BATCH_SIZE,\n",
    "        per_device_eval_batch_size=BATCH_SIZE * 2,\n",
    "        learning_rate=LEARNING_RATE,\n",
    "        weight_decay=WEIGHT_DECAY,\n",
    "        warmup_ratio=WARMUP_RATIO,\n",
    "        eval_strategy='epoch',\n",
    "        save_strategy='epoch',\n",
    "        save_total_limit=1,\n",
    "        load_best_model_at_end=True,\n",
    "        metric_for_best_model=METRIC_FOR_BEST,\n",
    "        greater_is_better=True,\n",
    "        logging_steps=100,\n",
    "        fp16=torch.cuda.is_available(),\n",
    "        seed=SEED,\n",
    "        report_to='none',\n",
    "        dataloader_num_workers=2,\n",
    "        save_safetensors=False,\n",
    "    )\n",
    "\n",
    "    trainer = Trainer(\n",
    "        model=model,\n",
    "        args=training_args,\n",
    "        train_dataset=train_ds,\n",
    "        eval_dataset=val_ds,\n",
    "        compute_metrics=compute_metrics,\n",
    "        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],\n",
    "    )\n",
    "\n",
    "    trainer.train()\n",
    "\n",
    "    val_results = trainer.evaluate(val_ds)\n",
    "    print(f\"  Val — Acc: {val_results['eval_accuracy']:.4f} | Macro-F1: {val_results['eval_macro_f1']:.4f}\")\n",
    "\n",
    "    test_output = trainer.predict(test_ds)\n",
    "    test_metrics = test_output.metrics\n",
    "    test_preds = np.argmax(test_output.predictions, axis=-1)\n",
    "    test_probs = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()\n",
    "    print(f\"  Test — Acc: {test_metrics['test_accuracy']:.4f} | Macro-F1: {test_metrics['test_macro_f1']:.4f}\")\n",
    "\n",
    "    val_output = trainer.predict(val_ds)\n",
    "    val_preds = np.argmax(val_output.predictions, axis=-1)\n",
    "    val_probs = torch.softmax(torch.tensor(val_output.predictions), dim=-1).numpy()\n",
    "\n",
    "    for split_name, texts, labels, preds, probs in [\n",
    "        ('test', test_texts, test_labels, test_preds, test_probs),\n",
    "        ('val',  val_texts,  val_labels,  val_preds,  val_probs),\n",
    "    ]:\n",
    "        pred_df = pd.DataFrame({\n",
    "            'text': texts, 'true_label': labels, 'pred_label': preds,\n",
    "            'prob_0': probs[:, 0], 'prob_1': probs[:, 1]\n",
    "        })\n",
    "        pred_df.to_csv(PRED_DIR / f\"07_{model_tag}_{dataset_tag}_{split_name}_predictions.csv\", index=False)\n",
    "\n",
    "    if SAVE_MODEL_WEIGHTS:\n",
    "        save_dir = CKPT_DIR / f\"07_{model_tag}_{dataset_tag}\" / 'best_model'\n",
    "        save_dir.mkdir(parents=True, exist_ok=True)\n",
    "        trainer.save_model(str(save_dir))\n",
    "        tokenizer.save_pretrained(str(save_dir))\n",
    "\n",
    "    cm = confusion_matrix(test_labels, test_preds).tolist()\n",
    "    t_elapsed = time.time() - t_start\n",
    "\n",
    "    result = {\n",
    "        'model_tag': model_tag, 'model_name': model_name, 'dataset': dataset_tag,\n",
    "        'val_accuracy': val_results['eval_accuracy'],\n",
    "        'val_macro_f1': val_results['eval_macro_f1'],\n",
    "        'val_weighted_f1': val_results['eval_weighted_f1'],\n",
    "        'val_binary_f1': val_results['eval_binary_f1'],\n",
    "        'test_accuracy': test_metrics['test_accuracy'],\n",
    "        'test_macro_f1': test_metrics['test_macro_f1'],\n",
    "        'test_weighted_f1': test_metrics['test_weighted_f1'],\n",
    "        'test_binary_f1': test_metrics['test_binary_f1'],\n",
    "        'test_precision': test_metrics['test_precision'],\n",
    "        'test_recall': test_metrics['test_recall'],\n",
    "        'confusion_matrix': json.dumps(cm),\n",
    "        'train_samples': len(train_texts),\n",
    "        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,\n",
    "        'max_length': MAX_LENGTH, 'time_seconds': round(t_elapsed, 1),\n",
    "    }\n",
    "\n",
    "    del model, trainer, train_ds, val_ds, test_ds\n",
    "    gc.collect()\n",
    "    if torch.cuda.is_available():\n",
    "        torch.cuda.empty_cache()\n",
    "    if run_tmp.exists():\n",
    "        shutil.rmtree(run_tmp)\n",
    "    clear_hf_cache()\n",
    "\n",
    "    print(f\"  Cleaned up. Disk free (tmp): {disk_free_gb(BIG_TMP):.1f} GB\")\n",
    "    return result"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "all_results = []\n",
    "summary_path = TABLE_DIR / '07_multi_backbone_binary_summary.csv'\n",
    "\n",
    "if summary_path.exists():\n",
    "    prev_df = pd.read_csv(summary_path)\n",
    "    all_results = prev_df.to_dict('records')\n",
    "    done_keys = set(zip(prev_df['model_tag'], prev_df['dataset']))\n",
    "    print(f\"Resuming: {len(done_keys)} runs already completed\")\n",
    "else:\n",
    "    done_keys = set()\n",
    "\n",
    "if TEMP_TRAIN_DIR.exists():\n",
    "    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)\n",
    "    TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)\n",
    "clear_hf_cache()\n",
    "\n",
    "total_runs = len(BACKBONES) * len(DATASETS)\n",
    "run_num = len(done_keys)\n",
    "\n",
    "for model_tag, model_name in BACKBONES.items():\n",
    "    for ds_tag, ds_paths in DATASETS.items():\n",
    "        if (model_tag, ds_tag) in done_keys:\n",
    "            print(f\"Skipping {model_tag} x {ds_tag} (done)\")\n",
    "            continue\n",
    "        run_num += 1\n",
    "        print(f\"\\n>>> Run {run_num}/{total_runs}\")\n",
    "\n",
    "        free_working = disk_free_gb('/kaggle/working')\n",
    "        if free_working < 3.0:\n",
    "            print(f\"WARNING: Only {free_working:.1f} GB free. Cleaning...\")\n",
    "            clear_hf_cache()\n",
    "            if free_working < 2.0:\n",
    "                raise RuntimeError(f\"Insufficient disk space ({free_working:.1f} GB). Aborting.\")\n",
    "\n",
    "        try:\n",
    "            result = train_and_evaluate(model_tag, model_name, ds_tag, ds_paths)\n",
    "            all_results.append(result)\n",
    "            pd.DataFrame(all_results).to_csv(summary_path, index=False)\n",
    "            print(f\"  Saved. {total_runs - run_num} runs remaining.\")\n",
    "        except Exception as e:\n",
    "            print(f\"  FAILED: {e}\")\n",
    "            if all_results:\n",
    "                pd.DataFrame(all_results).to_csv(summary_path, index=False)\n",
    "            if TEMP_TRAIN_DIR.exists():\n",
    "                shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)\n",
    "            clear_hf_cache()\n",
    "            raise\n",
    "\n",
    "if TEMP_TRAIN_DIR.exists():\n",
    "    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)\n",
    "clear_hf_cache()\n",
    "\n",
    "print(f\"\\n{'='*70}\")\n",
    "print(f\"All {total_runs} runs complete!\")\n",
    "print(f\"Disk free (working): {disk_free_gb('/kaggle/working'):.1f} GB\")\n",
    "print(f\"Disk free (tmp):     {disk_free_gb(BIG_TMP):.1f} GB\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "results_df = pd.read_csv(summary_path)\n",
    "display_cols = ['model_tag', 'dataset', 'test_accuracy', 'test_macro_f1', 'test_binary_f1', 'test_precision', 'test_recall', 'time_seconds']\n",
    "print(\"=\"*90)\n",
    "print(\"  MULTI-BACKBONE BINARY RESULTS (Test Set)\")\n",
    "print(\"=\"*90)\n",
    "print(results_df[display_cols].to_string(index=False, float_format='%.4f'))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "pivot = results_df.pivot_table(index='model_tag', columns='dataset', values='test_macro_f1', aggfunc='first')\n",
    "\n",
    "# Load previous baselines if available\n",
    "bb_path = TABLE_DIR / '05_baseline_banglabert_binary_summary_all_datasets.csv'\n",
    "if bb_path.exists():\n",
    "    bb_df = pd.read_csv(bb_path)\n",
    "    bb_row = {}\n",
    "    for _, row in bb_df.iterrows():\n",
    "        ds = row.get('dataset') or row.get('dataset_tag') or row.get('dataset_name')\n",
    "        f1 = row.get('test_macro_f1') or row.get('macro_f1') or row.get('eval_macro_f1')\n",
    "        if ds and f1:\n",
    "            bb_row[ds] = f1\n",
    "    if bb_row:\n",
    "        pivot.loc['banglabert (nb05)'] = bb_row\n",
    "\n",
    "fgm_path = TABLE_DIR / '06_fgm_banglabert_binary_summary_all_datasets.csv'\n",
    "if fgm_path.exists():\n",
    "    fgm_df = pd.read_csv(fgm_path)\n",
    "    fgm_row = {}\n",
    "    for _, row in fgm_df.iterrows():\n",
    "        ds = row.get('dataset') or row.get('dataset_tag') or row.get('dataset_name')\n",
    "        f1 = row.get('test_macro_f1') or row.get('macro_f1') or row.get('eval_macro_f1')\n",
    "        if ds and f1:\n",
    "            fgm_row[ds] = f1\n",
    "    if fgm_row:\n",
    "        pivot.loc['banglabert+fgm (nb06)'] = fgm_row\n",
    "\n",
    "pivot['mean'] = pivot.mean(axis=1)\n",
    "pivot = pivot.sort_values('mean', ascending=False)\n",
    "\n",
    "print(\"=\"*90)\n",
    "print(\"  MACRO-F1 COMPARISON — All Backbones x All Datasets (Test)\")\n",
    "print(\"=\"*90)\n",
    "print(pivot.to_string(float_format='%.4f'))\n",
    "print(f\"\\nBest backbone: {pivot['mean'].idxmax()} (mean={pivot['mean'].max():.4f})\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save confusion matrices\n",
    "cm_dict = {}\n",
    "for _, row in results_df.iterrows():\n",
    "    cm_dict[f\"{row['model_tag']}_{row['dataset']}\"] = json.loads(row['confusion_matrix'])\n",
    "with open(TABLE_DIR / '07_multi_backbone_binary_confusion_matrices.json', 'w') as f:\n",
    "    json.dump(cm_dict, f, indent=2)\n",
    "\n",
    "# Save metadata\n",
    "meta_cols = ['model_tag', 'model_name', 'dataset', 'train_samples', 'epochs', 'batch_size', 'lr', 'max_length', 'time_seconds']\n",
    "results_df[meta_cols].to_csv(LOG_DIR / '07_multi_backbone_run_metadata.csv', index=False)\n",
    "\n",
    "# Classification reports\n",
    "all_reports = []\n",
    "for _, row in results_df.iterrows():\n",
    "    pp = PRED_DIR / f\"07_{row['model_tag']}_{row['dataset']}_test_predictions.csv\"\n",
    "    if pp.exists():\n",
    "        pdf = pd.read_csv(pp)\n",
    "        rpt = classification_report(pdf['true_label'], pdf['pred_label'], output_dict=True, target_names=['Not Sarcastic', 'Sarcastic'])\n",
    "        for cls_name, metrics in rpt.items():\n",
    "            if isinstance(metrics, dict):\n",
    "                all_reports.append({'model_tag': row['model_tag'], 'dataset': row['dataset'], 'class': cls_name, **metrics})\n",
    "if all_reports:\n",
    "    pd.DataFrame(all_reports).to_csv(REPORT_DIR / '07_multi_backbone_binary_classification_reports.csv', index=False)\n",
    "\n",
    "print(\"All artifacts saved.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display confusion matrices\n",
    "for _, row in results_df.iterrows():\n",
    "    cm = np.array(json.loads(row['confusion_matrix']))\n",
    "    print(f\"\\n{row['model_tag']} x {row['dataset']}\")\n",
    "    print(f\"  Macro-F1: {row['test_macro_f1']:.4f} | Acc: {row['test_accuracy']:.4f}\")\n",
    "    print(f\"                  Pred=0   Pred=1\")\n",
    "    print(f\"    True=0     {cm[0,0]:>6}   {cm[0,1]:>6}\")\n",
    "    print(f\"    True=1     {cm[1,0]:>6}   {cm[1,1]:>6}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"Files produced in PROJECT/04_outputs:\")\n",
    "for p in sorted(PROJECT.rglob('07_*')):\n",
    "    if p.is_file() and 'predictions' in str(p):\n",
    "        print(f\"  {p.relative_to(PROJECT)}  ({p.stat().st_size / 1e6:.1f} MB)\")\n",
    "print(f\"\\nDisk free (working): {disk_free_gb('/kaggle/working'):.1f} GB\")\n",
    "print(f\"Disk free (tmp):     {disk_free_gb(BIG_TMP):.1f} GB\")\n",
    "print(\"\\n✅ Notebook finished without exceeding disk limits.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

# Save the notebook to a file
with open('07_multi_backbone_binarys.ipynb', 'w', encoding='utf-8') as f:
    json.dump(notebook_content, f, indent=2, ensure_ascii=False)

print("✅ Notebook saved as '07_multi_backbone_binary.ipynb'")

✅ Notebook saved as '07_multi_backbone_binary.ipynb'
